In [2]:
import torch
from transformers import AutoTokenizer, AutoModel

In [27]:
models = {
    "ncbi/MedCPT-Query-Encoder": {},
    "dmis-lab/biobert-v1.1": {},
    "NeuML/pubmedbert-base-embeddings": {},
}

queries = [
    "What cells express ACE2 and CD2 ?",
    "What cells don't express CD1 ?",
    "What cells is positive for Cedherin11 ?",
    "For which cells is CD109-1 present ?",
    "Which cells is negative for CD107a-LAMP-1 ?"
]

proteins = [
    'C5L2', 'Cadherin11', 'CCR10-1', 'CD101-BB27', 'CD102-ICAM-2', 'CD105',
       'CD106', 'CD107a-LAMP-1', 'CD109-1', 'CD110',
       'CD86-1', 'CD9-1', 'CD90-Thy1', 'CD94', 'CD95-Fas', 'CD98', 'HLA-A-B-C',
       'HLA-DR', 'IgD', 'IgGFc'
]

In [28]:
for model_name in models.keys():
    models[model_name]["model"] = AutoModel.from_pretrained(model_name)
    models[model_name]["tokenizer"] = AutoTokenizer.from_pretrained(model_name)

In [30]:
tokens = {}
for model_name in models.keys():
    print(f"--- Model : {model_name} ---")

    with torch.no_grad():
        # tokenize the queries
        encoded = models[model_name]["tokenizer"](
            queries, 
            truncation=True, 
            padding=True, 
            return_tensors='pt', 
            max_length=64,
        )

        # encode the queries (use the [CLS] last hidden states as the representations)
        embeds = models[model_name]["model"](**encoded).last_hidden_state[:, 0, :]
        print(f"Embeddings shape for model {model_name}: {embeds.shape}")
    print()

    for i, query in enumerate(queries):
        tokens = models[model_name]["tokenizer"].convert_ids_to_tokens(encoded["input_ids"][i])
        print(f"Tokens for query '{query}' in model {model_name}: {tokens}")
    print("\n\n")


--- Model : ncbi/MedCPT-Query-Encoder ---
Embeddings shape for model ncbi/MedCPT-Query-Encoder: torch.Size([5, 768])

Tokens for query 'What cells express ACE2 and CD2 ?' in model ncbi/MedCPT-Query-Encoder: ['[CLS]', 'what', 'cells', 'express', 'ace', '##2', 'and', 'cd2', '?', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Tokens for query 'What cells don't express CD1 ?' in model ncbi/MedCPT-Query-Encoder: ['[CLS]', 'what', 'cells', 'don', "'", 't', 'express', 'cd1', '?', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Tokens for query 'What cells is positive for Cedherin11 ?' in model ncbi/MedCPT-Query-Encoder: ['[CLS]', 'what', 'cells', 'is', 'positive', 'for', 'ced', '##her', '##in1', '##1', '?', '[SEP]', '[PAD]', '[PAD]']
Tokens for query 'For which cells is CD109-1 present ?' in model ncbi/MedCPT-Query-Encoder: ['[CLS]', 'for', 'which', 'cells', 'is', 'cd10', '##9', '-', '1', 'present', '?', '[SEP]', '[PAD]', '[PAD]']
Tokens for query 'Which cells is negative for CD107a-LAMP-1 ?' in m